# Chapter 1

### Importing image

```
# NOTE : Always resize image with PIL and then read the image with matplotlib

### Resizing image
from PIL import Image
img = Image.open("your_image.jpg") # Open the image
resized_img = img.resize((16, 16)) # Resize the image
resized_img.save("resized_image.jpg") # Save or display the resized image

### Reading image for manipulation
import matplotlib.pyplot as plt
data = plt.imread('image.jpg')
plt.imshow(data)
plt.show()
print(data.shape)
# Setting channel values to 0 intensity
data[:, :, 0] = 0 # red channel
data[:, :, 1] = 0 # green channel
data[:, :, 2] = 0 # blue channel
plt.imshow(data)
gray_image = data.mean(axis=2)

### Using OpenCV
import cv2
rgb_image = cv2.imread("input_image.jpg")
gray_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)
cv2.imwrite("output_image.jpg", gray_image)  # Save the grayscale image
cv2.imshow("Grayscale Image", gray_image)  # Display the grayscale image
```

### Convolution in python

```
# Problem with training neural network with Dense layer for images approach : we are assuming each pixel is independent. So we are using Dense layer. However, in fact, these pixels are related to their neighboring pixels, so if we flatten them, we will lose information about their relative neighboring pixels and their features. Is there any way to create features for a region of pixels? (Feature engineering on a region on pixels, just like how we calculate BMI by combining height and weight features)
# Solution : Convolution
# Assuming that the pixels are not independent and are related to their neighbors (Now you need convolution layers to also consider a pixel and the surrounding pixels. While Dense layer has 1 weight per pixel since it only takes 1 features, Convolution layer has 1 weight per kernel since it is taking neighboring pixels into consideration)
# Dense layers create more parameters on first layers, less parameters in last layers
# Convolutional layers create less parameters on first layers (which are more complex and expressive), so to read them, it creates more parameters in last layers
# Simple convolution (along 1 dimension) (Find the distinct features around the neighbors by comparing the differences among them)
array = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1]) # Original image
kernel = np.array([-1, 1]) # window scope that takes neighbors of pixels and slides along vertical or horizontal axis
conv = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0]) # Reduced image due to sum of product and kernel size and image (as window size increases by 1 , image size decreases by 1, you need to pad the reduced image with 0 to turn them back into original shape )
conv[0] = (kernel * array[0:2]).sum()
conv[1] = (kernel * array[1:3]).sum()
conv[2] = (kernel * array[2:4]).sum()
...
for ii in range(0,8,1): # Sliding from index 0 to inde 8, stepping 1 at a time (this step is known as sliding stride, you can increase it if you want to skip some sum of products of reduntant pixels)
    conv[ii] = (kernel * array[ii:ii+2]).sum() # Find
# You may add 0 at the first and the last index to increase the array size (This will be considered as zero-padding)
# 2D Convolution (along 2 dimensions)
kernel = np.array([[-1, 1], 
                [-1, 1]]) # A 2X2 kernel
conv = np.zeros((27, 27)) # This is a feature map, that contains features in the image (Initially this is empty, we will populate it with matrix multiplication of image window and kernel)
for ii in range(27): # Stride 1 step from left to right in row of image pixels
    for jj in range(27): # Stride 1 step from top to bottom in column of image pixels
        conv[ii, jj] = np.sum(image[ii:ii+2, jj:jj+2] * kernel) # Create a comparative value of surrounding pixels (convolved pixel result for the given kernel window)

# After convolution, output_size = (input_size - kernel_size + 2*padding_size)/strides + 1
# What is the size of the output for an input of size 256 by 256, with a kernel of size 4 by 4, padding of 1 and strides of 2?
input_size = 256 
kernel_size = 4 
padding_size = 1
strides = 2
output_size = (input_size - kernel_size + 2*padding_size)/strides + 1

# Reduce parameters : use pooling layer to pull the more distinguished pixel (brightest feature) in a selected region of pixels
result = np.zeros((im.shape[0]//2, im.shape[1]//2)) # it is the pooling result, initially all 0 matrix
for ii in range(result.shape[0]):
    for jj in range(result.shape[1]):
        result[ii, jj] = np.max(im[ii*2:ii*2+2, jj*2:jj*2+2]) # stride is 2 for each iteration, done on 2X2 window pool

```

### Training CNN

```

from keras.models import Sequential
model = Sequential()
from keras.layers import Dense
train_data.shape
# If you consider pixels are Independent, then use Dense Layer
model.add(Dense(10, activation='relu', input_shape=(784,)))
model.add(Dense(10, activation='relu'))
model.add(Dense(3, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
df = df.reshape((100, 784)) # (1 row = one sample image, 1 column = one pixel, total width*height for total pixels for total columns)
X= df.iloc[:, :-1]
y= df.iloc[:,-1]
from tensorflow.keras.utils import to_categorical # For categorical target variable, you need something like one-hot-encoding
y = to_categorical(data['target']) # Use this for one-hot-encoding if target is a class and it is a classification problem
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( X,y , random_state=104,test_size=0.25, shuffle=True)
model.fit(X_train, y_train, validation_split=0.2, epochs=3)
model.evaluate(X_test, y_test)

# If you consider pixels are not Independent, then use Convolution Layer
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Flatten
model = Sequential()
# Start with convolution layer that will create 10 feature maps
model.add(Conv2D(10, kernel_size=3, activation='relu', input_shape=(img_rows, img_cols, 1)), padding='same', strides=1, dilation_rate=2) # padding default is 'valid', and dilation helps aggregate information across multiple scales (2,2) pixels skips
model.add(MaxPool2D(pool_size=2)) # Pooling layer to reduce pixel parameters with a 2X2 window size pulling
model.add(Conv2D(15, kernel_size=3, activation='relu'))
model.add(MaxPool2D(2))
model.add(Flatten()) # Converts multi-dimensional neurons to single dimensional neurons
model.add(Dense(3, activation='softmax')) # The final output layer 
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_labels, validation_split=0.2, epochs=3, batch_size=10)
model.evaluate(test_data, test_labels, epochs=3)

```

# Chapter 2

### Convolution in image

<center><img src="images/02.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.04.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.05.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.03.png"  style="width: 400px, height: 300px;"/></center>

# Chapter 3

<center><img src="images/03.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.03.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.06.png"  style="width: 400px, height: 300px;"/></center>

### Maxpooling

<center><img src="images/03.05.png"  style="width: 400px, height: 300px;"/></center>
